In [ ]:
class AdvancedInformationTheoryFramework:
    """
    Comprehensive implementation of advanced information theory covering measure-theoretic
    foundations, f-divergences, algorithmic information theory, and information geometry
    with rigorous mathematical foundations and machine learning applications.
    """
    
    def __init__(self):
        self.information_measures = {}
        self.divergences = {}
        self.compression_algorithms = {}
        
    def demonstrate_information_measures_and_entropy(self):
        """
        Demonstrate Shannon entropy, differential entropy, mutual information,
        and advanced entropy measures including Rényi entropies.
        """
        print("=== Information Measures and Entropy Theory ===\n")
        
        # 1. Shannon entropy and basic measures
        print("1. Shannon Entropy and Basic Information Measures:")
        
        def shannon_entropy_discrete(p):
            """Compute Shannon entropy for discrete distribution"""
            p = np.array(p)
            p = p[p > 0]  # Remove zero probabilities
            return -np.sum(p * np.log2(p))
        
        def differential_entropy_gaussian(sigma):
            """Differential entropy of multivariate Gaussian"""
            if np.isscalar(sigma):
                return 0.5 * np.log2(2 * np.pi * np.e * sigma**2)
            else:
                # Multivariate case
                d = len(sigma) if sigma.ndim == 1 else sigma.shape[0]
                if sigma.ndim == 1:
                    det_sigma = np.prod(sigma)
                else:
                    det_sigma = np.linalg.det(sigma)
                return 0.5 * d * np.log2(2 * np.pi * np.e) + 0.5 * np.log2(det_sigma)
        
        def mutual_information_gaussian(sigma_xy, sigma_x, sigma_y):
            """Mutual information for joint Gaussian variables"""
            det_xy = np.linalg.det(sigma_xy)
            det_x = np.linalg.det(sigma_x) if sigma_x.ndim > 1 else sigma_x
            det_y = np.linalg.det(sigma_y) if sigma_y.ndim > 1 else sigma_y
            
            return 0.5 * np.log2((det_x * det_y) / det_xy)
        
        # Test discrete entropy
        distributions = {
            'uniform': [0.25, 0.25, 0.25, 0.25],
            'peaked': [0.7, 0.1, 0.1, 0.1],
            'binary': [0.5, 0.5, 0, 0]
        }
        
        print("Discrete Shannon Entropy:")
        for name, p in distributions.items():
            entropy = shannon_entropy_discrete(p)
            print(f"  {name}: H = {entropy:.3f} bits")
        
        # Test differential entropy
        print("\nDifferential Entropy (Gaussian):")
        sigmas = [0.5, 1.0, 2.0]
        for sigma in sigmas:
            h = differential_entropy_gaussian(sigma)
            print(f"  σ = {sigma}: h = {h:.3f} bits")
        
        # Multivariate case
        Sigma_2d = np.array([[1, 0.5], [0.5, 2]])
        h_2d = differential_entropy_gaussian(Sigma_2d)
        print(f"  2D Gaussian: h = {h_2d:.3f} bits")
        
        # 2. Rényi entropies
        print("\n2. Rényi Entropies:")
        
        def renyi_entropy(p, alpha):
            """Compute Rényi entropy of order α"""
            if alpha == 1:
                return shannon_entropy_discrete(p)
            elif alpha == 0:
                # Hartley entropy (log of support size)
                return np.log2(np.sum(np.array(p) > 0))
            elif alpha == np.inf:
                # Min-entropy
                return -np.log2(np.max(p))
            else:
                p = np.array(p)
                p = p[p > 0]
                return 1/(1-alpha) * np.log2(np.sum(p**alpha))
        
        # Test Rényi entropies
        p_test = [0.5, 0.3, 0.2]
        alphas = [0, 0.5, 1, 2, np.inf]
        
        print("Rényi Entropies for p = [0.5, 0.3, 0.2]:")
        for alpha in alphas:
            h_alpha = renyi_entropy(p_test, alpha)
            print(f"  H_{alpha}(X) = {h_alpha:.3f} bits")
        
        # 3. Mutual information estimation
        print("\n3. Mutual Information Estimation:")
        
        def estimate_mutual_information_kde(x, y, bins=20):
            """Estimate mutual information using histogram method"""
            # Joint histogram
            hist_xy, x_edges, y_edges = np.histogram2d(x, y, bins=bins, density=True)
            hist_x, _ = np.histogram(x, bins=x_edges, density=True)
            hist_y, _ = np.histogram(y, bins=y_edges, density=True)
            
            # Bin areas
            dx = x_edges[1] - x_edges[0]
            dy = y_edges[1] - y_edges[0]
            
            # Convert to probabilities
            p_xy = hist_xy * dx * dy
            p_x = hist_x * dx
            p_y = hist_y * dy
            
            # Compute MI
            mi = 0
            for i in range(len(p_x)):
                for j in range(len(p_y)):
                    if p_xy[i, j] > 1e-12 and p_x[i] > 1e-12 and p_y[j] > 1e-12:
                        mi += p_xy[i, j] * np.log2(p_xy[i, j] / (p_x[i] * p_y[j]))
            
            return mi
        
        def estimate_mutual_information_kraskov(x, y, k=3):
            """Estimate MI using Kraskov-Stögbauer-Grassberger method (simplified)"""
            from scipy.spatial.distance import cdist
            
            n = len(x)
            data = np.column_stack([x, y])
            
            # For each point, find distance to k-th nearest neighbor
            distances = cdist(data, data)
            np.fill_diagonal(distances, np.inf)
            
            # k-th nearest neighbor distances
            knn_distances = np.sort(distances, axis=1)[:, k-1]
            
            # Count neighbors within this distance in marginal spaces
            mi_sum = 0
            for i in range(n):
                eps = knn_distances[i]
                
                # Count neighbors in x and y margins
                n_x = np.sum(np.abs(x - x[i]) < eps)
                n_y = np.sum(np.abs(y - y[i]) < eps)
                
                if n_x > 0 and n_y > 0:
                    mi_sum += np.log(n) + np.log(k) - np.log(n_x) - np.log(n_y)
            
            return mi_sum / n / np.log(2)  # Convert to bits
        
        # Generate correlated data
        np.random.seed(42)
        n_samples = 1000
        correlations = [0, 0.3, 0.6, 0.9]
        
        print("Mutual Information Estimation:")
        for rho in correlations:
            # Generate correlated Gaussian data
            mean = [0, 0]
            cov = [[1, rho], [rho, 1]]
            data = np.random.multivariate_normal(mean, cov, n_samples)
            x, y = data[:, 0], data[:, 1]
            
            # Theoretical MI for Gaussian
            mi_theoretical = -0.5 * np.log2(1 - rho**2)
            
            # Empirical estimates
            mi_histogram = estimate_mutual_information_kde(x, y)
            mi_kraskov = estimate_mutual_information_kraskov(x, y)
            
            print(f"  ρ = {rho}: Theory = {mi_theoretical:.3f}, "
                  f"Histogram = {mi_histogram:.3f}, KSG = {mi_kraskov:.3f}")
        
        # 4. Information-theoretic quantities
        print("\n4. Information-Theoretic Quantities:")
        
        def conditional_entropy(joint_p):
            """Compute conditional entropy H(Y|X) from joint distribution"""
            joint_p = np.array(joint_p)
            marginal_x = np.sum(joint_p, axis=1)
            
            h_y_given_x = 0
            for i in range(joint_p.shape[0]):
                if marginal_x[i] > 0:
                    conditional_p = joint_p[i, :] / marginal_x[i]
                    h_y_given_x += marginal_x[i] * shannon_entropy_discrete(conditional_p)
            
            return h_y_given_x
        
        def total_correlation(joint_p):
            """Compute total correlation (multi-information)"""
            # For 2D case: TC = H(X) + H(Y) - H(X,Y)
            marginal_x = np.sum(joint_p, axis=1)
            marginal_y = np.sum(joint_p, axis=0)
            
            h_x = shannon_entropy_discrete(marginal_x)
            h_y = shannon_entropy_discrete(marginal_y)
            h_xy = shannon_entropy_discrete(joint_p.flatten())
            
            return h_x + h_y - h_xy
        
        # Test on joint distribution
        joint_dist = np.array([[0.4, 0.1], [0.1, 0.4]])
        
        marginal_x = np.sum(joint_dist, axis=1)
        marginal_y = np.sum(joint_dist, axis=0)
        
        h_x = shannon_entropy_discrete(marginal_x)
        h_y = shannon_entropy_discrete(marginal_y)
        h_xy = shannon_entropy_discrete(joint_dist.flatten())
        h_y_given_x = conditional_entropy(joint_dist)
        tc = total_correlation(joint_dist)
        mi = h_y - h_y_given_x
        
        print("Joint Distribution Analysis:")
        print(f"  H(X) = {h_x:.3f}")
        print(f"  H(Y) = {h_y:.3f}")
        print(f"  H(X,Y) = {h_xy:.3f}")
        print(f"  H(Y|X) = {h_y_given_x:.3f}")
        print(f"  I(X;Y) = {mi:.3f}")
        print(f"  Total Correlation = {tc:.3f}")
        
        # Verify information identity: H(X,Y) = H(X) + H(Y|X)
        identity_check = abs(h_xy - (h_x + h_y_given_x))
        print(f"  Identity check |H(X,Y) - H(X) - H(Y|X)| = {identity_check:.6f}")
        
        # Visualize results
        plt.figure(figsize=(15, 12))
        
        # Entropy comparison for different distributions
        plt.subplot(3, 4, 1)
        dist_names = list(distributions.keys())
        entropies = [shannon_entropy_discrete(distributions[name]) for name in dist_names]
        
        plt.bar(dist_names, entropies, alpha=0.7)
        plt.ylabel('Entropy (bits)')
        plt.title('Shannon Entropy Comparison')
        plt.grid(True, alpha=0.3)
        
        # Rényi entropy spectrum
        plt.subplot(3, 4, 2)
        alpha_range = np.linspace(0.1, 5, 50)
        renyi_values = []
        
        for alpha in alpha_range:
            if alpha != 1:
                renyi_values.append(renyi_entropy(p_test, alpha))
            else:
                renyi_values.append(shannon_entropy_discrete(p_test))
        
        plt.plot(alpha_range, renyi_values, 'b-', linewidth=2)
        plt.axvline(x=1, color='red', linestyle='--', label='Shannon (α=1)')
        plt.xlabel('α')
        plt.ylabel('H_α(X) (bits)')
        plt.title('Rényi Entropy Spectrum')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # Differential entropy vs variance
        plt.subplot(3, 4, 3)
        sigma_range = np.linspace(0.1, 3, 50)
        diff_entropies = [differential_entropy_gaussian(s) for s in sigma_range]
        
        plt.plot(sigma_range, diff_entropies, 'g-', linewidth=2)
        plt.xlabel('Standard Deviation σ')
        plt.ylabel('h(X) (bits)')
        plt.title('Gaussian Differential Entropy')
        plt.grid(True, alpha=0.3)
        
        # Mutual information vs correlation
        plt.subplot(3, 4, 4)
        rho_range = np.linspace(0, 0.95, 20)
        mi_theoretical = [-0.5 * np.log2(1 - rho**2) for rho in rho_range]
        
        plt.plot(rho_range, mi_theoretical, 'r-', linewidth=2, label='Theoretical')
        
        # Add empirical points
        rho_empirical = correlations
        mi_empirical = []
        for rho in rho_empirical:
            mean = [0, 0]
            cov = [[1, rho], [rho, 1]]
            data = np.random.multivariate_normal(mean, cov, 500)
            x, y = data[:, 0], data[:, 1]
            mi_emp = estimate_mutual_information_kde(x, y)
            mi_empirical.append(mi_emp)
        
        plt.plot(rho_empirical, mi_empirical, 'bo', label='Empirical')
        plt.xlabel('Correlation ρ')
        plt.ylabel('I(X;Y) (bits)')
        plt.title('Mutual Information vs Correlation')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # Information diagram
        plt.subplot(3, 4, 5)
        
        # Venn diagram style visualization
        circle_x = plt.Circle((0.3, 0.5), 0.25, alpha=0.3, color='blue', label='H(X)')
        circle_y = plt.Circle((0.7, 0.5), 0.25, alpha=0.3, color='red', label='H(Y)')
        
        plt.gca().add_patch(circle_x)
        plt.gca().add_patch(circle_y)
        
        # Add text annotations
        plt.text(0.15, 0.5, f'H(X|Y)\n{h_x - mi:.2f}', ha='center', va='center')
        plt.text(0.5, 0.5, f'I(X;Y)\n{mi:.2f}', ha='center', va='center')
        plt.text(0.85, 0.5, f'H(Y|X)\n{h_y_given_x:.2f}', ha='center', va='center')
        
        plt.xlim(0, 1)
        plt.ylim(0.2, 0.8)
        plt.title('Information Diagram')
        plt.axis('equal')
        plt.axis('off')
        
        # Entropy rate for sequences
        plt.subplot(3, 4, 6)
        
        # Simulate Markov chain
        def markov_entropy_rate(transition_matrix, stationary_dist):
            """Compute entropy rate of Markov chain"""
            H_rate = 0
            for i in range(len(stationary_dist)):
                if stationary_dist[i] > 0:
                    row_entropy = shannon_entropy_discrete(transition_matrix[i, :])
                    H_rate += stationary_dist[i] * row_entropy
            return H_rate
        
        # Create transition matrix
        P = np.array([[0.7, 0.3], [0.4, 0.6]])
        # Compute stationary distribution
        eigenvals, eigenvecs = np.linalg.eig(P.T)
        stationary = eigenvecs[:, np.argmax(eigenvals)].real
        stationary = stationary / np.sum(stationary)
        
        h_rate = markov_entropy_rate(P, stationary)
        
        # Show transition matrix
        im = plt.imshow(P, cmap='Blues', aspect='auto')
        plt.colorbar(im)
        plt.title(f'Markov Chain\nEntropy Rate: {h_rate:.3f}')
        plt.xlabel('To State')
        plt.ylabel('From State')
        
        for i in range(2):
            for j in range(2):
                plt.text(j, i, f'{P[i,j]:.2f}', ha='center', va='center')
        
        # Maximum entropy distributions
        plt.subplot(3, 4, 7)
        
        # Compare distributions with same mean but different entropies
        x_range = np.linspace(-3, 3, 100)
        
        # Gaussian (maximum entropy for given variance)
        gaussian_pdf = (1/np.sqrt(2*np.pi)) * np.exp(-0.5 * x_range**2)
        
        # Laplace distribution
        laplace_pdf = 0.5 * np.exp(-np.abs(x_range))
        
        # Uniform (maximum entropy for given support)
        uniform_pdf = np.where(np.abs(x_range) <= 1, 0.5, 0)
        
        plt.plot(x_range, gaussian_pdf, 'b-', label='Gaussian', linewidth=2)
        plt.plot(x_range, laplace_pdf, 'r--', label='Laplace', linewidth=2)
        plt.plot(x_range, uniform_pdf, 'g:', label='Uniform', linewidth=2)
        
        plt.xlabel('x')
        plt.ylabel('Density')
        plt.title('Maximum Entropy Distributions')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # Information bottleneck illustration
        plt.subplot(3, 4, 8)
        
        # Simulate information bottleneck trade-off
        compression_rates = np.linspace(0, 1, 20)
        prediction_accuracy = 1 - np.exp(-5 * compression_rates)  # Increasing with compression
        
        plt.plot(compression_rates, prediction_accuracy, 'bo-', alpha=0.8)
        plt.xlabel('Compression Rate I(T;X)')
        plt.ylabel('Prediction Accuracy I(T;Y)')
        plt.title('Information Bottleneck Trade-off')
        plt.grid(True, alpha=0.3)
        
        # Cross-entropy illustration
        plt.subplot(3, 4, 9)
        
        # True vs predicted distributions
        p_true = np.array([0.7, 0.2, 0.1])
        p_pred_good = np.array([0.65, 0.25, 0.1])
        p_pred_bad = np.array([0.3, 0.3, 0.4])
        
        categories = ['A', 'B', 'C']
        x_pos = np.arange(len(categories))
        
        width = 0.25
        plt.bar(x_pos - width, p_true, width, label='True', alpha=0.7)
        plt.bar(x_pos, p_pred_good, width, label='Good Pred', alpha=0.7)
        plt.bar(x_pos + width, p_pred_bad, width, label='Bad Pred', alpha=0.7)
        
        # Compute cross-entropies
        ce_good = -np.sum(p_true * np.log2(p_pred_good))
        ce_bad = -np.sum(p_true * np.log2(p_pred_bad))
        
        plt.xlabel('Category')
        plt.ylabel('Probability')
        plt.title(f'Cross-Entropy\nGood: {ce_good:.2f}, Bad: {ce_bad:.2f}')
        plt.legend()
        plt.xticks(x_pos, categories)
        plt.grid(True, alpha=0.3)
        
        # Conditional mutual information
        plt.subplot(3, 4, 10)
        
        # Generate 3D Gaussian data
        mean_3d = [0, 0, 0]
        # Create dependency: Z affects both X and Y
        cov_3d = [[1, 0.6, 0.7], [0.6, 1, 0.8], [0.7, 0.8, 1]]
        data_3d = np.random.multivariate_normal(mean_3d, cov_3d, 1000)
        x_3d, y_3d, z_3d = data_3d[:, 0], data_3d[:, 1], data_3d[:, 2]
        
        # Estimate I(X;Y) and I(X;Y|Z)
        mi_xy = estimate_mutual_information_kde(x_3d, y_3d)
        
        # For conditional MI, we'd need more sophisticated estimation
        # Here we show the concept
        plt.scatter(x_3d[:100], y_3d[:100], c=z_3d[:100], alpha=0.6, cmap='viridis')
        plt.colorbar(label='Z value')
        plt.xlabel('X')
        plt.ylabel('Y')
        plt.title(f'Conditional Independence\nI(X;Y) = {mi_xy:.2f}')
        plt.grid(True, alpha=0.3)
        
        # Information flow in Markov chain
        plt.subplot(3, 4, 11)
        
        # Simulate Markov chain X → Y → Z
        n_chain = 1000
        X_chain = np.random.randn(n_chain)
        Y_chain = 0.8 * X_chain + 0.6 * np.random.randn(n_chain)
        Z_chain = 0.7 * Y_chain + 0.7 * np.random.randn(n_chain)
        
        # Estimate mutual informations
        mi_xy_chain = estimate_mutual_information_kde(X_chain, Y_chain)
        mi_xz_chain = estimate_mutual_information_kde(X_chain, Z_chain)
        mi_yz_chain = estimate_mutual_information_kde(Y_chain, Z_chain)
        
        mis = [mi_xy_chain, mi_xz_chain, mi_yz_chain]
        labels = ['I(X;Y)', 'I(X;Z)', 'I(Y;Z)']
        colors = ['blue', 'red', 'green']
        
        plt.bar(labels, mis, color=colors, alpha=0.7)
        plt.ylabel('Mutual Information')
        plt.title('Data Processing Inequality\nI(X;Z) ≤ I(X;Y)')
        plt.grid(True, alpha=0.3)
        
        # Verify data processing inequality
        if mi_xz_chain <= mi_xy_chain:
            plt.text(0.5, max(mis) * 0.8, '✓ Inequality holds', 
                    ha='center', fontsize=12, color='green')
        else:
            plt.text(0.5, max(mis) * 0.8, '✗ Inequality violated', 
                    ha='center', fontsize=12, color='red')
        
        # Information theory identities
        plt.subplot(3, 4, 12)
        
        # Verify various information identities
        identities = {
            'Chain rule': abs(h_xy - (h_x + h_y_given_x)),
            'MI symmetry': abs(mi - (h_y - h_y_given_x)),
            'Subadditivity': h_xy - (h_x + h_y)  # Should be ≤ 0
        }
        
        identity_names = list(identities.keys())
        values = list(identities.values())
        colors = ['green' if abs(v) < 1e-6 else 'red' for v in values]
        
        plt.bar(identity_names, values, color=colors, alpha=0.7)
        plt.ylabel('Identity Value')
        plt.title('Information Theory Identities')
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        return {
            'entropy_analysis': {
                'discrete_entropies': entropies,
                'differential_entropies': diff_entropies,
                'renyi_spectrum': renyi_values
            },
            'mutual_information': {
                'theoretical': mi_theoretical,
                'empirical_histogram': mi_empirical,
                'correlations': correlations
            },
            'information_quantities': {
                'joint_entropy': h_xy,
                'conditional_entropy': h_y_given_x,
                'mutual_information': mi,
                'total_correlation': tc
            }
        }

# Information Theory: Mathematical Foundations for Machine Learning

## Mathematical Overview

Information theory, founded by Claude Shannon, provides the mathematical framework for quantifying, processing, and transmitting information. It forms the theoretical foundation for modern machine learning, combining probability theory, measure theory, and functional analysis to understand the fundamental limits of learning, compression, and communication.

## Learning Objectives

By the end of this notebook, you will understand:

**1. Measure-Theoretic Information Theory:**
- Entropy and divergences on probability spaces
- Differential entropy and continuous information
- Information-theoretic inequalities and extremal principles
- Optimal transport and Wasserstein information geometry

**2. Advanced Information Measures:**
- Rényi entropies and their applications
- f-divergences and their unified theory  
- Mutual information and conditional independence
- Information bottleneck principle and rate-distortion theory

**3. Algorithmic Information Theory:**
- Kolmogorov complexity and algorithmic randomness
- Minimum description length principle
- Universal coding and prediction
- Connection to PAC-Bayesian theory

**4. Information Geometry:**
- Statistical manifolds and Fisher information metric
- Natural gradients and information-geometric optimization
- Exponential families and dually flat manifolds
- Applications to variational inference and neural networks

## Mathematical Prerequisites

- Measure theory and integration
- Probability theory and stochastic processes
- Functional analysis and optimization theory
- Differential geometry and manifold theory

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.special import xlogy
from sklearn.datasets import load_iris, make_classification
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_selection import mutual_info_classif
import pandas as pd
from collections import Counter
import itertools
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

## 1. Measure-Theoretic Foundations of Information Theory

### Definition 1.1: Shannon Entropy on Measure Spaces
Let (Ω, ℱ, μ) be a probability space and X: Ω → 𝒳 a random variable. The **Shannon entropy** is:
$$H(X) = -\int_{\mathcal{X}} p_X(x) \log p_X(x) \, d\nu(x)$$

where pₓ is the density of μₓ = μ ∘ X⁻¹ with respect to reference measure ν.

**Properties:**
- **Non-negativity**: H(X) ≥ 0 with equality iff X is deterministic
- **Concavity**: H is concave in the distribution of X
- **Maximum entropy**: H(X) ≤ log |𝒳| for finite 𝒳, achieved by uniform distribution

### Definition 1.2: Differential Entropy
For absolutely continuous random variable X with density p on ℝⁿ:
$$h(X) = -\int_{\mathbb{R}^n} p(x) \log p(x) \, dx$$

**Critical differences from discrete entropy:**
- Can be negative
- Not invariant under change of variables: h(AX) = h(X) + log|det(A)|
- **Translation invariant**: h(X + c) = h(X)

### Theorem 1.1: Maximum Entropy Principle
Among all distributions on ℝⁿ with given second moment E[XXᵀ] = Σ, the multivariate Gaussian maximizes differential entropy:
$$h(X) \leq \frac{1}{2} \log[(2\pi e)^n \det(\Sigma)]$$

**Proof sketch**: Use calculus of variations with Lagrange multipliers.

### Definition 1.3: f-Divergences
For convex function f with f(1) = 0, the **f-divergence** is:
$$D_f(P \| Q) = \int f\left(\frac{dP}{dQ}\right) dQ$$

**Special cases:**
- **KL divergence**: f(t) = t log t
- **Total variation**: f(t) = ½|t - 1|
- **Hellinger distance**: f(t) = (√t - 1)²
- **χ² divergence**: f(t) = (t - 1)²

### Theorem 1.2: Pinsker's Inequality
For any two probability measures P, Q:
$$\|P - Q\|_{TV} \leq \sqrt{\frac{1}{2} D_{KL}(P \| Q)}$$

where ‖P - Q‖ₜᵥ is the total variation distance.

### Definition 1.4: Information-Theoretic Inequalities
**Data Processing Inequality**: For Markov chain X → Y → Z:
$$I(X; Z) \leq I(X; Y)$$

**Submodularity**: For random variables X, Y, Z:
$$I(X; Y|Z) + I(X; Z) \geq I(X; YZ)$$

**Log-Sobolev Inequality**: For μ = N(0, I):
$$\text{Ent}_\mu(f^2) \leq 2 \int |\nabla f|^2 d\mu$$

where Entμ(g) = ∫g log g dμ - (∫g dμ) log(∫g dμ).

In [ ]:
# Information theory fundamentals
print("Information Theory Key Concepts:")
print("")
print("Information Content: I(x) = -log₂(P(x)) bits")
print("• Rare events carry more information")
print("• Certain events (P=1) carry no information")
print("")
print("Entropy: H(X) = -∑ P(x) log₂ P(x)")
print("• Measures average information content")
print("• Maximum when distribution is uniform")
print("• Minimum (0) when distribution is deterministic")

def entropy(probabilities, base=2):
    """Calculate entropy of a probability distribution"""
    # Remove zero probabilities to avoid log(0)
    probabilities = np.array(probabilities)
    probabilities = probabilities[probabilities > 0]
    
    if base == 2:
        return -np.sum(probabilities * np.log2(probabilities))
    elif base == np.e:
        return -np.sum(probabilities * np.log(probabilities))
    else:
        return -np.sum(probabilities * np.log(probabilities) / np.log(base))

def information_content(probability, base=2):
    """Calculate information content of an event"""
    if probability <= 0:
        return float('inf')
    
    if base == 2:
        return -np.log2(probability)
    elif base == np.e:
        return -np.log(probability)
    else:
        return -np.log(probability) / np.log(base)

# Demonstrate entropy with different distributions
def demonstrate_entropy():
    """Show how entropy varies with different probability distributions"""
    
    distributions = {
        'Uniform (2 outcomes)': [0.5, 0.5],
        'Uniform (4 outcomes)': [0.25, 0.25, 0.25, 0.25],
        'Uniform (8 outcomes)': [0.125] * 8,
        'Skewed': [0.9, 0.1],
        'Very skewed': [0.99, 0.01],
        'Deterministic': [1.0, 0.0],
        'Three-way uniform': [1/3, 1/3, 1/3],
        'Three-way skewed': [0.8, 0.15, 0.05]
    }
    
    print(f"\nEntropy Examples:")
    print(f"{'Distribution':<20} {'Probabilities':<25} {'Entropy (bits)':<15} {'Max Entropy':<15}")
    print("-" * 80)
    
    for name, probs in distributions.items():
        H = entropy(probs)
        max_H = np.log2(len(probs))  # Maximum entropy for n outcomes
        
        probs_str = str([f"{p:.2f}" for p in probs])
        print(f"{name:<20} {probs_str:<25} {H:<15.3f} {max_H:<15.3f}")
    
    return distributions

distributions = demonstrate_entropy()

# Visualize entropy
def visualize_entropy():
    """Visualize how entropy changes with probability distributions"""
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Plot 1: Binary entropy function
    ax1 = axes[0, 0]
    p = np.linspace(0.001, 0.999, 1000)
    binary_entropy = entropy(np.column_stack([p, 1-p]))
    
    ax1.plot(p, binary_entropy, 'b-', linewidth=2)
    ax1.axhline(y=1, color='red', linestyle='--', alpha=0.7, label='Maximum entropy (1 bit)')
    ax1.axvline(x=0.5, color='red', linestyle='--', alpha=0.7)
    ax1.set_xlabel('Probability of outcome 1')
    ax1.set_ylabel('Entropy (bits)')
    ax1.set_title('Binary Entropy Function')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Information content vs probability
    ax2 = axes[0, 1]
    probs = np.linspace(0.001, 1, 1000)
    info_content = [information_content(p) for p in probs]
    
    ax2.plot(probs, info_content, 'g-', linewidth=2)
    ax2.set_xlabel('Probability')
    ax2.set_ylabel('Information Content (bits)')
    ax2.set_title('Information Content vs Probability')
    ax2.set_yscale('log')
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Entropy for different distributions
    ax3 = axes[1, 0]
    selected_dists = ['Uniform (2 outcomes)', 'Skewed', 'Very skewed', 'Deterministic']
    entropies = [entropy(distributions[name]) for name in selected_dists]
    colors = ['blue', 'orange', 'red', 'gray']
    
    bars = ax3.bar(range(len(selected_dists)), entropies, color=colors, alpha=0.7)
    ax3.set_xticks(range(len(selected_dists)))
    ax3.set_xticklabels(selected_dists, rotation=45, ha='right')
    ax3.set_ylabel('Entropy (bits)')
    ax3.set_title('Entropy of Different Distributions')
    ax3.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, entropy_val in zip(bars, entropies):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{entropy_val:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # Plot 4: Entropy vs number of outcomes (uniform distribution)
    ax4 = axes[1, 1]
    n_outcomes = range(2, 17)
    uniform_entropies = [np.log2(n) for n in n_outcomes]
    
    ax4.plot(n_outcomes, uniform_entropies, 'mo-', linewidth=2, markersize=6)
    ax4.set_xlabel('Number of Equally Likely Outcomes')
    ax4.set_ylabel('Entropy (bits)')
    ax4.set_title('Maximum Entropy vs Number of Outcomes')
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

visualize_entropy()

print(f"\nKey Insights about Entropy:")
print(f"• Maximum entropy occurs with uniform distributions")
print(f"• Entropy = 0 for deterministic outcomes")
print(f"• Binary entropy peaks at p = 0.5 with H = 1 bit")
print(f"• More outcomes → higher maximum possible entropy")
print(f"• Entropy measures the minimum bits needed to encode outcomes")

### ML Application: Decision Trees and Information Gain

Information gain is used in decision trees to choose the best splits by measuring how much a split reduces entropy.

In [ ]:
# Information gain in decision trees
def information_gain(parent_labels, split_labels_list, base=2):
    """Calculate information gain from a split"""
    
    # Calculate parent entropy
    parent_counts = Counter(parent_labels)
    parent_probs = [count / len(parent_labels) for count in parent_counts.values()]
    parent_entropy = entropy(parent_probs, base)
    
    # Calculate weighted average of child entropies
    total_samples = len(parent_labels)
    weighted_child_entropy = 0
    
    for child_labels in split_labels_list:
        if len(child_labels) == 0:
            continue
            
        child_counts = Counter(child_labels)
        child_probs = [count / len(child_labels) for count in child_counts.values()]
        child_entropy = entropy(child_probs, base)
        
        weight = len(child_labels) / total_samples
        weighted_child_entropy += weight * child_entropy
    
    # Information gain = reduction in entropy
    return parent_entropy - weighted_child_entropy

# Create a simple dataset for decision tree demonstration
def create_decision_tree_dataset():
    """Create a simple dataset for decision tree analysis"""
    
    # Weather dataset for playing tennis
    data = {
        'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rainy', 'Rainy', 'Rainy', 'Overcast',
                   'Sunny', 'Sunny', 'Rainy', 'Sunny', 'Overcast', 'Overcast', 'Rainy'],
        'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool',
                       'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'],
        'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal',
                    'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
        'Windy': ['False', 'True', 'False', 'False', 'False', 'True', 'True',
                 'False', 'False', 'False', 'True', 'True', 'False', 'True'],
        'PlayTennis': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes',
                      'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
    }
    
    return pd.DataFrame(data)

# Analyze information gain for different features
def analyze_information_gain():
    """Analyze information gain for decision tree features"""
    
    df = create_decision_tree_dataset()
    target = df['PlayTennis'].values
    
    print("Decision Tree Dataset:")
    print(df.head(10))
    print(f"\nDataset shape: {df.shape}")
    
    # Calculate parent entropy
    target_counts = Counter(target)
    target_probs = [count / len(target) for count in target_counts.values()]
    parent_entropy = entropy(target_probs)
    
    print(f"\nParent Entropy (PlayTennis):")
    print(f"Class distribution: {dict(target_counts)}")
    print(f"Entropy: {parent_entropy:.3f} bits")
    
    # Calculate information gain for each feature
    features = ['Outlook', 'Temperature', 'Humidity', 'Windy']
    gains = {}
    
    print(f"\nInformation Gain Analysis:")
    print(f"{'Feature':<12} {'Values':<25} {'Information Gain':<15}")
    print("-" * 55)
    
    for feature in features:
        feature_values = df[feature].unique()
        
        # Split target by feature values
        split_targets = []
        for value in feature_values:
            mask = df[feature] == value
            split_targets.append(target[mask])
        
        # Calculate information gain
        gain = information_gain(target, split_targets)
        gains[feature] = gain
        
        values_str = ', '.join(feature_values)
        print(f"{feature:<12} {values_str:<25} {gain:<15.3f}")
    
    # Find best feature
    best_feature = max(gains, key=gains.get)
    print(f"\nBest feature for root split: {best_feature} (gain = {gains[best_feature]:.3f})")
    
    return df, gains

df, gains = analyze_information_gain()

# Visualize the decision tree construction process
def visualize_decision_tree_construction():
    """Visualize how decision trees use information gain"""
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Plot 1: Information gain comparison
    ax1 = axes[0, 0]
    features = list(gains.keys())
    gain_values = list(gains.values())
    colors = ['red' if feature == 'Outlook' else 'lightblue' for feature in features]
    
    bars = ax1.bar(features, gain_values, color=colors, alpha=0.7)
    ax1.set_ylabel('Information Gain (bits)')
    ax1.set_title('Information Gain by Feature')
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar, gain in zip(bars, gain_values):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{gain:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # Plot 2: Class distribution by best feature (Outlook)
    ax2 = axes[0, 1]
    outlook_groups = df.groupby(['Outlook', 'PlayTennis']).size().unstack(fill_value=0)
    outlook_groups.plot(kind='bar', ax=ax2, color=['lightcoral', 'lightgreen'], alpha=0.7)
    ax2.set_xlabel('Outlook')
    ax2.set_ylabel('Count')
    ax2.set_title('Class Distribution by Outlook')
    ax2.legend(title='Play Tennis')
    ax2.tick_params(axis='x', rotation=45)
    
    # Plot 3: Entropy reduction visualization
    ax3 = axes[1, 0]
    
    # Calculate entropy for each outlook value
    outlook_entropies = []
    outlook_weights = []
    outlook_values = df['Outlook'].unique()
    
    for outlook in outlook_values:
        subset = df[df['Outlook'] == outlook]['PlayTennis']
        counts = Counter(subset)
        probs = [count / len(subset) for count in counts.values()]
        ent = entropy(probs)
        outlook_entropies.append(ent)
        outlook_weights.append(len(subset) / len(df))
    
    # Parent entropy
    parent_counts = Counter(df['PlayTennis'])
    parent_probs = [count / len(df) for count in parent_counts.values()]
    parent_ent = entropy(parent_probs)
    
    # Weighted average child entropy
    weighted_child_ent = sum(w * e for w, e in zip(outlook_weights, outlook_entropies))
    
    categories = ['Parent\n(Before Split)', 'Weighted Average\n(After Split)']
    entropies = [parent_ent, weighted_child_ent]
    colors = ['red', 'blue']
    
    bars = ax3.bar(categories, entropies, color=colors, alpha=0.7)
    ax3.set_ylabel('Entropy (bits)')
    ax3.set_title('Entropy Before and After Split')
    ax3.grid(True, alpha=0.3, axis='y')
    
    # Add value labels and gain
    for bar, ent in zip(bars, entropies):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{ent:.3f}', ha='center', va='bottom', fontweight='bold')
    
    gain_arrow = parent_ent - weighted_child_ent
    ax3.annotate(f'Information Gain = {gain_arrow:.3f}', 
                xy=(0.5, (parent_ent + weighted_child_ent)/2),
                xytext=(1.5, (parent_ent + weighted_child_ent)/2),
                arrowprops=dict(arrowstyle='<->', color='green', lw=2),
                fontsize=12, fontweight='bold', color='green')
    
    # Plot 4: Decision tree structure
    ax4 = axes[1, 1]
    ax4.text(0.5, 0.9, 'Decision Tree Structure', ha='center', va='center', 
             fontsize=14, fontweight='bold', transform=ax4.transAxes)
    
    # Simple text representation of tree
    tree_text = [
        "Root: Outlook",
        "├─ Sunny: Humidity",
        "│  ├─ High → No",
        "│  └─ Normal → Yes",
        "├─ Overcast → Yes",
        "└─ Rainy: Windy",
        "   ├─ True → No",
        "   └─ False → Yes"
    ]
    
    for i, line in enumerate(tree_text):
        ax4.text(0.1, 0.8 - i*0.08, line, ha='left', va='center', 
                fontsize=10, family='monospace', transform=ax4.transAxes)
    
    ax4.axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_decision_tree_construction()

# Build actual decision tree with sklearn
def build_sklearn_decision_tree():
    """Build and visualize decision tree using sklearn"""
    
    # Encode categorical variables
    from sklearn.preprocessing import LabelEncoder
    
    df_encoded = df.copy()
    label_encoders = {}
    
    for column in df.columns:
        if df[column].dtype == 'object':
            le = LabelEncoder()
            df_encoded[column] = le.fit_transform(df[column])
            label_encoders[column] = le
    
    X = df_encoded.drop('PlayTennis', axis=1)
    y = df_encoded['PlayTennis']
    
    # Build decision tree with entropy criterion
    dt = DecisionTreeClassifier(criterion='entropy', random_state=42, max_depth=3)
    dt.fit(X, y)
    
    # Display tree performance
    accuracy = dt.score(X, y)
    print(f"\nDecision Tree Results:")
    print(f"Training accuracy: {accuracy:.3f}")
    print(f"Tree depth: {dt.get_depth()}")
    print(f"Number of leaves: {dt.get_n_leaves()}")
    
    # Feature importance (based on information gain)
    feature_names = X.columns
    importances = dt.feature_importances_
    
    print(f"\nFeature Importances:")
    for feature, importance in zip(feature_names, importances):
        print(f"{feature}: {importance:.3f}")
    
    return dt, X, y, feature_names

dt, X, y, feature_names = build_sklearn_decision_tree()

print(f"\nKey Insights about Information Gain in Decision Trees:")
print(f"• Information gain measures reduction in entropy after a split")
print(f"• Best splits maximize information gain")
print(f"• Pure nodes (entropy = 0) don't need further splitting")
print(f"• Information gain can be biased toward features with more values")
print(f"• Gain ratio addresses this bias by normalizing with split information")

## 2. Advanced Information Measures and Rate-Distortion Theory

### Definition 2.1: Mutual Information and Conditional Independence
For random variables X, Y with joint distribution P_{X,Y}:
$$I(X; Y) = D_{KL}(P_{X,Y} \| P_X \otimes P_Y) = \iint \log \frac{p_{X,Y}(x,y)}{p_X(x)p_Y(y)} p_{X,Y}(x,y) \, dx \, dy$$

**Conditional mutual information**:
$$I(X; Y | Z) = \mathbb{E}_Z [D_{KL}(P_{X,Y|Z} \| P_{X|Z} \otimes P_{Y|Z})]$$

### Definition 2.2: Rényi Entropy and Divergence
The **Rényi entropy** of order α > 0, α ≠ 1 is:
$$H_\alpha(X) = \frac{1}{1-\alpha} \log \int p(x)^\alpha \, dx$$

**Properties:**
- H₁(X) = H(X) (Shannon entropy as limit α → 1)
- H₀(X) = log |supp(p)| (Hartley entropy)
- H∞(X) = -log max p(x) (min-entropy)
- Hα is decreasing in α

**Rényi divergence**:
$$D_\alpha(P \| Q) = \frac{1}{\alpha-1} \log \int p(x)^\alpha q(x)^{1-\alpha} \, dx$$

### Theorem 2.1: Information Bottleneck Principle
Consider the optimization problem:
$$\min_{p(t|x)} I(X; T) - \beta I(T; Y)$$

where T is a **bottleneck variable**. The optimal solution satisfies:
$$p(t|x) = \frac{p(t)}{Z(x,\beta)} \exp\left(\beta \sum_y p(y|t) \log \frac{p(y|t)}{p(y)}\right)$$

**Applications**: Feature extraction, representation learning, deep learning theory.

### Definition 2.3: Rate-Distortion Function
For source X and distortion measure d(x,x̂), the **rate-distortion function** is:
$$R(D) = \min_{p(\hat{x}|x): \mathbb{E}[d(X,\hat{X})] \leq D} I(X; \hat{X})$$

**Interpretation**: Minimum information rate needed to represent X with distortion ≤ D.

### Theorem 2.2: Blahut-Arimoto Algorithm
The rate-distortion function can be computed iteratively:
$$p^{(t+1)}(\hat{x}|x) = \frac{p^{(t)}(\hat{x})}{Z^{(t)}(x)} \exp(-\lambda d(x,\hat{x}))$$
$$p^{(t+1)}(\hat{x}) = \sum_x p(x) p^{(t+1)}(\hat{x}|x)$$

where λ is chosen to satisfy the distortion constraint.

### Definition 2.4: Channel Capacity
For discrete memoryless channel with transition probabilities p(y|x), the **capacity** is:
$$C = \max_{p(x)} I(X; Y)$$

**Shannon's Channel Coding Theorem**: For rates R < C, there exist codes with arbitrarily low error probability.

### Theorem 2.3: Concavity and Convexity Properties
- **Entropy is concave** in the distribution
- **Mutual information is convex** in the conditional distribution p(y|x) for fixed input distribution
- **Rate-distortion function is convex** in distortion D
- **KL divergence is convex** in both arguments

### Application 2.1: Information-Theoretic Feature Selection
**Mutual Information Maximization**:
$$\max_{S \subseteq \{1,\ldots,d\}, |S|=k} I(X_S; Y)$$

**Conditional Mutual Information** (accounting for redundancy):
$$I(X_i; Y | X_{S_{i-1}}) = H(Y | X_{S_{i-1}}) - H(Y | X_{S_{i-1}}, X_i)$$

where S_{i-1} are previously selected features.

In [ ]:
# Mutual information concepts
print("Mutual Information:")
print("")
print("Definition: I(X;Y) = H(X) - H(X|Y) = H(Y) - H(Y|X)")
print("")
print("Alternative form: I(X;Y) = ∑∑ p(x,y) log[p(x,y)/(p(x)p(y))]")
print("")
print("Properties:")
print("• I(X;Y) ≥ 0 (non-negative)")
print("• I(X;Y) = 0 if and only if X and Y are independent")
print("• I(X;Y) = I(Y;X) (symmetric)")
print("• I(X;X) = H(X) (self-information equals entropy)")

def mutual_information(X, Y, base=2):
    """
    Calculate mutual information between two discrete random variables
    """
    # Create joint probability distribution
    joint_counts = Counter(zip(X, Y))
    total_samples = len(X)
    
    # Marginal distributions
    X_counts = Counter(X)
    Y_counts = Counter(Y)
    
    # Calculate mutual information
    mi = 0.0
    for (x, y), joint_count in joint_counts.items():
        p_xy = joint_count / total_samples
        p_x = X_counts[x] / total_samples
        p_y = Y_counts[y] / total_samples
        
        if p_xy > 0 and p_x > 0 and p_y > 0:
            if base == 2:
                mi += p_xy * np.log2(p_xy / (p_x * p_y))
            elif base == np.e:
                mi += p_xy * np.log(p_xy / (p_x * p_y))
            else:
                mi += p_xy * np.log(p_xy / (p_x * p_y)) / np.log(base)
    
    return mi

def conditional_entropy(X, Y, base=2):
    """
    Calculate conditional entropy H(X|Y)
    """
    Y_counts = Counter(Y)
    total_samples = len(Y)
    
    conditional_ent = 0.0
    
    for y_val, y_count in Y_counts.items():
        # Get X values for this Y value
        X_given_y = [X[i] for i in range(len(X)) if Y[i] == y_val]
        
        if len(X_given_y) > 0:
            # Calculate entropy of X given Y=y_val
            X_given_y_counts = Counter(X_given_y)
            X_given_y_probs = [count / len(X_given_y) for count in X_given_y_counts.values()]
            H_X_given_y = entropy(X_given_y_probs, base)
            
            # Weight by P(Y=y_val)
            p_y = y_count / total_samples
            conditional_ent += p_y * H_X_given_y
    
    return conditional_ent

# Demonstrate mutual information with different relationships
def demonstrate_mutual_information():
    """Show mutual information for different types of relationships"""
    
    np.random.seed(42)
    n_samples = 1000
    
    # 1. Independent variables
    X_indep = np.random.choice([0, 1], size=n_samples)
    Y_indep = np.random.choice([0, 1], size=n_samples)
    
    # 2. Perfectly dependent variables
    X_dep = np.random.choice([0, 1], size=n_samples)
    Y_dep = X_dep.copy()  # Y = X
    
    # 3. Partially dependent variables
    X_partial = np.random.choice([0, 1], size=n_samples)
    # Y has 80% chance of being same as X
    noise = np.random.random(n_samples) < 0.2
    Y_partial = X_partial.copy()
    Y_partial[noise] = 1 - Y_partial[noise]
    
    # 4. Categorical variables with multiple values
    X_multi = np.random.choice(['A', 'B', 'C'], size=n_samples)
    # Y depends on X with some pattern
    Y_multi = []
    for x in X_multi:
        if x == 'A':
            y = np.random.choice(['X', 'Y'], p=[0.8, 0.2])
        elif x == 'B':
            y = np.random.choice(['X', 'Y'], p=[0.5, 0.5])
        else:  # x == 'C'
            y = np.random.choice(['X', 'Y'], p=[0.2, 0.8])
        Y_multi.append(y)
    
    # Calculate mutual information for each case
    cases = [
        (X_indep, Y_indep, "Independent"),
        (X_dep, Y_dep, "Perfect dependence (Y=X)"),
        (X_partial, Y_partial, "Partial dependence (80% match)"),
        (X_multi, Y_multi, "Categorical with pattern")
    ]
    
    print(f"\nMutual Information Analysis:")
    print(f"{'Relationship':<30} {'H(X)':<8} {'H(Y)':<8} {'H(X|Y)':<8} {'I(X;Y)':<8}")
    print("-" * 70)
    
    results = {}
    
    for X, Y, description in cases:
        # Calculate entropies and mutual information
        H_X = entropy([np.mean(np.array(X) == val) for val in np.unique(X)])
        H_Y = entropy([np.mean(np.array(Y) == val) for val in np.unique(Y)])
        H_X_given_Y = conditional_entropy(X, Y)
        I_XY = mutual_information(X, Y)
        
        results[description] = {
            'H_X': H_X,
            'H_Y': H_Y,
            'H_X_given_Y': H_X_given_Y,
            'I_XY': I_XY,
            'X': X,
            'Y': Y
        }
        
        print(f"{description:<30} {H_X:<8.3f} {H_Y:<8.3f} {H_X_given_Y:<8.3f} {I_XY:<8.3f}")
    
    return results

mi_results = demonstrate_mutual_information()

# Visualize mutual information relationships
def visualize_mutual_information():
    """Visualize different types of variable relationships"""
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    cases = list(mi_results.keys())
    
    for i, case in enumerate(cases):
        ax = axes[i//2, i%2]
        
        X = mi_results[case]['X']
        Y = mi_results[case]['Y']
        I_XY = mi_results[case]['I_XY']
        
        # Create contingency table
        unique_X = sorted(np.unique(X))
        unique_Y = sorted(np.unique(Y))
        
        # Create joint probability matrix
        joint_probs = np.zeros((len(unique_X), len(unique_Y)))
        
        for i_x, x_val in enumerate(unique_X):
            for i_y, y_val in enumerate(unique_Y):
                joint_probs[i_x, i_y] = np.mean((np.array(X) == x_val) & (np.array(Y) == y_val))
        
        # Plot heatmap
        im = ax.imshow(joint_probs, cmap='Blues', aspect='auto')
        
        # Add text annotations
        for i_x in range(len(unique_X)):
            for i_y in range(len(unique_Y)):
                ax.text(i_y, i_x, f'{joint_probs[i_x, i_y]:.3f}',
                        ha='center', va='center', fontweight='bold')
        
        ax.set_xticks(range(len(unique_Y)))
        ax.set_yticks(range(len(unique_X)))
        ax.set_xticklabels(unique_Y)
        ax.set_yticklabels(unique_X)
        ax.set_xlabel('Y')
        ax.set_ylabel('X')
        ax.set_title(f'{case}\nI(X;Y) = {I_XY:.3f} bits')
        
        # Add colorbar
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    plt.show()

visualize_mutual_information()

# Information-theoretic measures comparison
def compare_information_measures():
    """Compare different information-theoretic measures"""
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: Information measures comparison
    ax1 = axes[0]
    
    measures = ['H(X)', 'H(Y)', 'H(X|Y)', 'I(X;Y)']
    
    case_names = list(mi_results.keys())
    x_pos = np.arange(len(case_names))
    width = 0.2
    
    for i, measure in enumerate(measures):
        values = []
        for case in case_names:
            if measure == 'H(X)':
                values.append(mi_results[case]['H_X'])
            elif measure == 'H(Y)':
                values.append(mi_results[case]['H_Y'])
            elif measure == 'H(X|Y)':
                values.append(mi_results[case]['H_X_given_Y'])
            else:  # I(X;Y)
                values.append(mi_results[case]['I_XY'])
        
        ax1.bar(x_pos + i*width, values, width, label=measure, alpha=0.8)
    
    ax1.set_xlabel('Relationship Type')
    ax1.set_ylabel('Information (bits)')
    ax1.set_title('Information-Theoretic Measures Comparison')
    ax1.set_xticks(x_pos + width * 1.5)
    ax1.set_xticklabels([case.split()[0] for case in case_names], rotation=45)
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Plot 2: Mutual information vs correlation strength
    ax2 = axes[1]
    
    # Generate data with different correlation strengths
    correlations = np.linspace(0, 0.95, 20)
    mi_values = []
    
    np.random.seed(42)
    for corr in correlations:
        # Generate correlated binary variables
        X = np.random.choice([0, 1], size=1000)
        # Y depends on X with given correlation
        Y = X.copy()
        flip_indices = np.random.choice(1000, size=int(1000 * (1-corr)), replace=False)
        Y[flip_indices] = 1 - Y[flip_indices]
        
        mi = mutual_information(X, Y)
        mi_values.append(mi)
    
    ax2.plot(correlations, mi_values, 'bo-', linewidth=2, markersize=6)
    ax2.set_xlabel('Correlation Strength')
    ax2.set_ylabel('Mutual Information (bits)')
    ax2.set_title('Mutual Information vs Correlation Strength')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

compare_information_measures()

print(f"\nKey Insights about Mutual Information:")
print(f"• MI = 0 for independent variables")
print(f"• Higher MI indicates stronger dependence")
print(f"• MI(X;Y) = H(X) when Y completely determines X")
print(f"• MI is symmetric: I(X;Y) = I(Y;X)")
print(f"• MI can detect non-linear relationships (unlike correlation)")
print(f"• Used in feature selection to find informative features")

## 3. Algorithmic Information Theory and Compression

### Definition 3.1: Kolmogorov Complexity
The **Kolmogorov complexity** K(x) of a string x is the length of the shortest program that outputs x:
$$K(x) = \min\{|p| : U(p) = x\}$$

where U is a universal Turing machine.

**Properties:**
- **Invariance**: K_U(x) ≤ K_V(x) + c for universal machines U, V
- **Uncomputability**: K is not computable
- **Randomness**: x is random if K(x) ≥ |x| - c

### Theorem 3.1: Algorithmic Probability and Solomonoff Induction
The **algorithmic probability** of x is:
$$P(x) = \sum_{p: U(p)=x} 2^{-|p|}$$

**Solomonoff prior**: Provides optimal prediction for any computable sequence.

### Definition 3.2: Minimum Description Length (MDL)
For model class ℳ and data D, choose model minimizing:
$$L(M) + L(D|M)$$

where L(M) is the description length of model M and L(D|M) is the data encoding length given M.

**Two-part codes**: Total description length = Model complexity + Data fit

### Theorem 3.2: MDL and Statistical Consistency
Under regularity conditions, MDL estimator converges to the true model as n → ∞.

### Definition 3.3: Universal Coding
A **universal code** for source class ℱ achieves:
$$\mathbb{E}[\ell(X)] \leq H(X) + R_n(X)$$

where R_n(X) → 0 as n → ∞ for all X ∈ ℱ.

**Lempel-Ziv coding**: Universal for all stationary ergodic sources.

### Application 3.1: Neural Network Compression
**Weight quantization with information theory**:
$$\min_Q \mathbb{E}_{w \sim p}[d(w, Q(w))] + \lambda H(Q)$$

where Q is quantization function and H(Q) measures code complexity.

### Definition 3.4: PAC-Bayesian Information Theory
For prior π and posterior ρ on hypothesis space ℋ:
$$\mathbb{P}\left[\mathcal{R}(ρ) \leq \hat{\mathcal{R}}(ρ) + \sqrt{\frac{D_{KL}(ρ \| π) + \ln(2\sqrt{m}/δ)}{2(m-1)}}\right] \geq 1-δ$$

**Interpretation**: KL divergence between posterior and prior acts as complexity penalty.

### Theorem 3.3: Normalized Maximum Likelihood (NML)
The **NML distribution** for model class ℳ_n is:
$$P_{NML}(x^n) = \frac{p(x^n | \hat{\theta}(x^n))}{\sum_{y^n} p(y^n | \hat{\theta}(y^n))}$$

where $\hat{\theta}(x^n)$ is the MLE for x^n.

**Regret**: NML minimizes worst-case log-loss regret over ℳ_n.

### Application 3.2: Variational Information Bottleneck
**Parametric information bottleneck**:
$$\mathcal{L}_{VIB} = \mathbb{E}_{p(x,y)}[D_{KL}(q_\phi(z|x) \| p(z))] - \beta \mathbb{E}_{q_\phi(z|x)}[\log p_\theta(y|z)]$$

**Interpretation**: Balance between compression (first term) and prediction (second term).

In [ ]:
# KL divergence and cross-entropy
print("KL Divergence and Cross-Entropy:")
print("")
print("KL Divergence: D_KL(P||Q) = ∑ P(x) log[P(x)/Q(x)]")
print("• Measures how P differs from Q")
print("• Always non-negative: D_KL(P||Q) ≥ 0")
print("• Zero if and only if P = Q (almost everywhere)")
print("• Not symmetric: D_KL(P||Q) ≠ D_KL(Q||P)")
print("")
print("Cross-Entropy: H(P,Q) = -∑ P(x) log Q(x)")
print("• H(P,Q) = H(P) + D_KL(P||Q)")
print("• Minimizing cross-entropy = minimizing KL divergence")
print("• Common loss function in classification")

def kl_divergence(P, Q, base=2):
    """
    Calculate KL divergence D_KL(P||Q)
    P, Q: probability distributions (must sum to 1)
    """
    P = np.array(P)
    Q = np.array(Q)
    
    # Avoid division by zero and log(0)
    epsilon = 1e-10
    P = np.clip(P, epsilon, 1)
    Q = np.clip(Q, epsilon, 1)
    
    if base == 2:
        return np.sum(P * np.log2(P / Q))
    elif base == np.e:
        return np.sum(P * np.log(P / Q))
    else:
        return np.sum(P * np.log(P / Q) / np.log(base))

def cross_entropy(P, Q, base=2):
    """
    Calculate cross-entropy H(P,Q)
    """
    P = np.array(P)
    Q = np.array(Q)
    
    epsilon = 1e-10
    Q = np.clip(Q, epsilon, 1)
    
    if base == 2:
        return -np.sum(P * np.log2(Q))
    elif base == np.e:
        return -np.sum(P * np.log(Q))
    else:
        return -np.sum(P * np.log(Q) / np.log(base))

def js_divergence(P, Q):
    """
    Calculate Jensen-Shannon divergence (symmetric version of KL)
    """
    P = np.array(P)
    Q = np.array(Q)
    M = 0.5 * (P + Q)
    
    return 0.5 * kl_divergence(P, M) + 0.5 * kl_divergence(Q, M)

# Demonstrate KL divergence with different distributions
def demonstrate_kl_divergence():
    """Show KL divergence for different distribution pairs"""
    
    # Define various probability distributions
    distributions = {
        'Uniform': [0.25, 0.25, 0.25, 0.25],
        'Skewed': [0.7, 0.2, 0.08, 0.02],
        'Peaked': [0.1, 0.8, 0.08, 0.02],
        'Bimodal': [0.4, 0.1, 0.1, 0.4],
        'Extreme': [0.97, 0.01, 0.01, 0.01]
    }
    
    print(f"\nKL Divergence Analysis:")
    print(f"Comparing all distributions to Uniform distribution")
    print(f"")
    print(f"{'Distribution':<12} {'Probabilities':<25} {'D_KL(P||Uniform)':<15} {'D_KL(Uniform||P)':<15}")
    print("-" * 75)
    
    uniform = distributions['Uniform']
    
    kl_results = {}
    
    for name, dist in distributions.items():
        kl_p_to_uniform = kl_divergence(dist, uniform)
        kl_uniform_to_p = kl_divergence(uniform, dist)
        
        kl_results[name] = {
            'distribution': dist,
            'kl_to_uniform': kl_p_to_uniform,
            'kl_from_uniform': kl_uniform_to_p
        }
        
        dist_str = '[' + ', '.join([f'{p:.2f}' for p in dist]) + ']'
        print(f"{name:<12} {dist_str:<25} {kl_p_to_uniform:<15.3f} {kl_uniform_to_p:<15.3f}")
    
    return distributions, kl_results

distributions, kl_results = demonstrate_kl_divergence()

# Cross-entropy in classification
def demonstrate_cross_entropy_loss():
    """Demonstrate cross-entropy loss in classification context"""
    
    # Simulate classification scenario
    # True labels (one-hot encoded)
    true_labels = np.array([
        [1, 0, 0],  # Class 0
        [0, 1, 0],  # Class 1  
        [0, 0, 1],  # Class 2
        [1, 0, 0],  # Class 0
        [0, 1, 0]   # Class 1
    ])
    
    # Different prediction scenarios
    predictions = {
        'Perfect': np.array([
            [1.0, 0.0, 0.0],
            [0.0, 1.0, 0.0],
            [0.0, 0.0, 1.0],
            [1.0, 0.0, 0.0],
            [0.0, 1.0, 0.0]
        ]),
        'Confident': np.array([
            [0.9, 0.08, 0.02],
            [0.05, 0.9, 0.05],
            [0.02, 0.08, 0.9],
            [0.85, 0.1, 0.05],
            [0.1, 0.85, 0.05]
        ]),
        'Uncertain': np.array([
            [0.6, 0.3, 0.1],
            [0.2, 0.6, 0.2],
            [0.15, 0.25, 0.6],
            [0.55, 0.35, 0.1],
            [0.25, 0.55, 0.2]
        ]),
        'Wrong': np.array([
            [0.1, 0.8, 0.1],  # Predicts class 1, true is 0
            [0.7, 0.2, 0.1],  # Predicts class 0, true is 1
            [0.4, 0.5, 0.1],  # Predicts class 1, true is 2
            [0.2, 0.7, 0.1],  # Predicts class 1, true is 0
            [0.6, 0.3, 0.1]   # Predicts class 0, true is 1
        ])
    }
    
    print(f"\nCross-Entropy Loss in Classification:")
    print(f"{'Scenario':<12} {'Avg Cross-Entropy':<18} {'Avg KL Divergence':<18} {'Accuracy':<10}")
    print("-" * 60)
    
    ce_results = {}
    
    for scenario, preds in predictions.items():
        # Calculate cross-entropy for each sample
        ce_losses = []
        kl_divs = []
        correct = 0
        
        for i in range(len(true_labels)):
            true_dist = true_labels[i]
            pred_dist = preds[i]
            
            ce = cross_entropy(true_dist, pred_dist, base=np.e)  # Use natural log for ML
            kl = kl_divergence(true_dist, pred_dist, base=np.e)
            
            ce_losses.append(ce)
            kl_divs.append(kl)
            
            # Check if prediction is correct
            if np.argmax(pred_dist) == np.argmax(true_dist):
                correct += 1
        
        avg_ce = np.mean(ce_losses)
        avg_kl = np.mean(kl_divs)
        accuracy = correct / len(true_labels)
        
        ce_results[scenario] = {
            'cross_entropy': avg_ce,
            'kl_divergence': avg_kl,
            'accuracy': accuracy,
            'predictions': preds
        }
        
        print(f"{scenario:<12} {avg_ce:<18.3f} {avg_kl:<18.3f} {accuracy:<10.3f}")
    
    return ce_results

ce_results = demonstrate_cross_entropy_loss()

# Visualize KL divergence and cross-entropy
def visualize_kl_divergence():
    """Visualize KL divergence and cross-entropy relationships"""
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Plot 1: Distribution comparison
    ax1 = axes[0, 0]
    
    x_labels = ['A', 'B', 'C', 'D']
    width = 0.15
    x_pos = np.arange(len(x_labels))
    
    colors = ['blue', 'red', 'green', 'purple', 'orange']
    
    for i, (name, dist) in enumerate(distributions.items()):
        ax1.bar(x_pos + i*width, dist, width, label=name, 
               color=colors[i], alpha=0.7)
    
    ax1.set_xlabel('Outcome')
    ax1.set_ylabel('Probability')
    ax1.set_title('Probability Distributions')
    ax1.set_xticks(x_pos + width * 2)
    ax1.set_xticklabels(x_labels)
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Plot 2: KL divergences (asymmetric)
    ax2 = axes[0, 1]
    
    dist_names = list(distributions.keys())
    kl_to_uniform = [kl_results[name]['kl_to_uniform'] for name in dist_names]
    kl_from_uniform = [kl_results[name]['kl_from_uniform'] for name in dist_names]
    
    x_pos = np.arange(len(dist_names))
    width = 0.35
    
    ax2.bar(x_pos - width/2, kl_to_uniform, width, label='D_KL(P||Uniform)', alpha=0.7)
    ax2.bar(x_pos + width/2, kl_from_uniform, width, label='D_KL(Uniform||P)', alpha=0.7)
    
    ax2.set_xlabel('Distribution')
    ax2.set_ylabel('KL Divergence (bits)')
    ax2.set_title('KL Divergence Asymmetry')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(dist_names, rotation=45)
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Plot 3: Cross-entropy loss in classification
    ax3 = axes[1, 0]
    
    scenarios = list(ce_results.keys())
    ce_values = [ce_results[scenario]['cross_entropy'] for scenario in scenarios]
    accuracies = [ce_results[scenario]['accuracy'] for scenario in scenarios]
    
    # Create double y-axis plot
    color = 'tab:red'
    ax3.set_xlabel('Prediction Scenario')
    ax3.set_ylabel('Cross-Entropy Loss', color=color)
    bars1 = ax3.bar(scenarios, ce_values, alpha=0.7, color=color)
    ax3.tick_params(axis='y', labelcolor=color)
    ax3.set_xticklabels(scenarios, rotation=45)
    
    ax3_twin = ax3.twinx()
    color = 'tab:blue'
    ax3_twin.set_ylabel('Accuracy', color=color)
    line = ax3_twin.plot(scenarios, accuracies, 'o-', color=color, linewidth=2, markersize=8)
    ax3_twin.tick_params(axis='y', labelcolor=color)
    
    ax3.set_title('Cross-Entropy Loss vs Accuracy')
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: KL divergence vs prediction confidence
    ax4 = axes[1, 1]
    
    # Vary prediction confidence for a true label [1, 0, 0]
    confidences = np.linspace(0.34, 0.99, 50)  # Start from 1/3 (uniform)
    true_label = [1, 0, 0]
    
    kl_values = []
    ce_values = []
    
    for conf in confidences:
        # Prediction: [conf, (1-conf)/2, (1-conf)/2]
        pred = [conf, (1-conf)/2, (1-conf)/2]
        
        kl = kl_divergence(true_label, pred, base=np.e)
        ce = cross_entropy(true_label, pred, base=np.e)
        
        kl_values.append(kl)
        ce_values.append(ce)
    
    ax4.plot(confidences, kl_values, 'r-', linewidth=2, label='KL Divergence')
    ax4.plot(confidences, ce_values, 'b-', linewidth=2, label='Cross-Entropy')
    
    ax4.set_xlabel('Prediction Confidence for True Class')
    ax4.set_ylabel('Loss Value')
    ax4.set_title('Loss vs Prediction Confidence')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    ax4.set_yscale('log')
    
    plt.tight_layout()
    plt.show()

visualize_kl_divergence()

print(f"\nKey Insights about KL Divergence and Cross-Entropy:")
print(f"• KL divergence is asymmetric: D_KL(P||Q) ≠ D_KL(Q||P)")
print(f"• Cross-entropy = entropy + KL divergence")
print(f"• Minimizing cross-entropy minimizes KL divergence")
print(f"• Lower cross-entropy often correlates with higher accuracy")
print(f"• Cross-entropy loss encourages confident, correct predictions")
print(f"• Jensen-Shannon divergence is symmetric version of KL")

## 4. Information Geometry and Neural Networks

### Definition 4.1: Statistical Manifold
A **statistical manifold** is a Riemannian manifold (ℳ, g) where:
- ℳ = {p(x|θ) : θ ∈ Θ} is a family of probability distributions
- g is the Fisher information metric: g_{ij}(θ) = E[∂ᵢ log p(X|θ) ∂ⱼ log p(X|θ)]

**Properties:**
- Invariant under sufficient statistics
- Natural Riemannian structure for statistical inference
- Geodesics represent efficient paths between distributions

### Definition 4.2: Fisher Information Metric
For parametric family {p(x|θ)}, the **Fisher information matrix** is:
$$I_{ij}(\theta) = -\mathbb{E}\left[\frac{\partial^2 \log p(X|\theta)}{\partial \theta_i \partial \theta_j}\right] = \mathbb{E}\left[\frac{\partial \log p(X|\theta)}{\partial \theta_i} \frac{\partial \log p(X|\theta)}{\partial \theta_j}\right]$$

**Cramér-Rao bound**: For unbiased estimator θ̂:
$$\text{Cov}(\hat{\theta}) \succeq I(\theta)^{-1}$$

### Theorem 4.1: Natural Gradient
The **natural gradient** with respect to Fisher metric is:
$$\tilde{\nabla}_\theta f(\theta) = I(\theta)^{-1} \nabla_\theta f(\theta)$$

**Properties:**
- Invariant under reparameterization
- Steepest descent direction in information geometry
- Reduces to Newton's method for exponential families

### Definition 4.3: Exponential Families and Dual Coordinates
An **exponential family** has the form:
$$p(x|\eta) = h(x) \exp(\eta^T T(x) - A(\eta))$$

**Dual coordinates**:
- Natural parameters: η (expectation parameters)
- Mean parameters: μ = ∇A(η) = E[T(X)]

**Dually flat structure**: (η, μ) provide flat coordinate systems connected by Legendre transform.

### Theorem 4.2: Amari's α-Connections
The **α-connection** ∇^(α) on statistical manifold satisfies:
$$\Gamma^{(α)}_{ij,k} = \mathbb{E}\left[\left(\frac{1-\alpha}{2} \frac{\partial^2 \log p}{\partial \theta_i \partial \theta_j} + \frac{1+\alpha}{2} \frac{\partial \log p}{\partial \theta_i} \frac{\partial \log p}{\partial \theta_j}\right) \frac{\partial \log p}{\partial \theta_k}\right]$$

**Special cases**:
- α = 1: e-connection (exponential, +1-flat)
- α = -1: m-connection (mixture, -1-flat)  
- α = 0: Levi-Civita connection

### Application 4.1: Natural Gradient in Neural Networks
For neural network with parameters θ, the natural gradient update is:
$$\theta_{t+1} = \theta_t - \alpha G(\theta_t)^{-1} \nabla_\theta L(\theta_t)$$

where G(θ) is the Fisher information matrix of the predictive distribution.

**K-FAC approximation**: Block-diagonal approximation to Fisher information for computational efficiency.

### Definition 4.4: Information Bottleneck and Deep Learning
**Deep Information Bottleneck**: Each layer ℓ satisfies:
$$\min_{p(t_\ell|t_{\ell-1})} I(T_\ell; T_{\ell-1}) - \beta I(T_\ell; Y)$$

**Tishby's hypothesis**: Deep networks undergo two phases:
1. **Fitting phase**: Increase I(T; Y) 
2. **Compression phase**: Decrease I(T; X) while preserving I(T; Y)

### Theorem 4.3: Variational Inference and Information Geometry
The **ELBO** in variational inference:
$$\mathcal{L}(q) = \mathbb{E}_q[\log p(x,z)] - \mathbb{E}_q[\log q(z)] = \log p(x) - D_{KL}(q(z) \| p(z|x))$$

**Natural gradient for variational inference**:
$$\nabla_{\text{nat}} \mathcal{L} = \mathbb{E}_q[(\nabla \log q(z))(\nabla \log q(z))^T]^{-1} \nabla \mathcal{L}$$

### Application 4.2: Optimal Transport and Wasserstein Information
**Wasserstein-2 distance**: For μ, ν ∈ 𝒫₂(ℝᵈ):
$$W_2(\mu, \nu) = \inf_{\gamma \in \Pi(\mu,\nu)} \left(\int \|x-y\|^2 d\gamma(x,y)\right)^{1/2}$$

**Wasserstein information**: Replaces KL divergence in information bottleneck:
$$\min_{q(z|x)} W_2(q(z|x), p(z)) - \beta I_W(Z; Y)$$

where I_W uses Wasserstein distance instead of KL divergence.

In [ ]:
# Information theory in neural networks
print("Information Theory in Neural Networks:")
print("")
print("Key Concepts:")
print("• Information Bottleneck: Networks compress input while preserving task-relevant info")
print("• Activation Information: Mutual information between layers and inputs/outputs")
print("• Generalization: Related to information compression in hidden layers")
print("• Loss Functions: Cross-entropy, KL divergence guide learning")

# Simple neural network with information analysis
class InformationAnalysisNetwork:
    """Simple neural network with information-theoretic analysis"""
    
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        
        # Initialize weights
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.1
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, output_dim) * 0.1
        self.b2 = np.zeros(output_dim)
        
        # Store activations for analysis
        self.activations = {}
    
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -250, 250)))
    
    def softmax(self, x):
        exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=-1, keepdims=True)
    
    def forward(self, X):
        # Hidden layer
        z1 = X @ self.W1 + self.b1
        a1 = self.sigmoid(z1)
        
        # Output layer
        z2 = a1 @ self.W2 + self.b2
        a2 = self.softmax(z2)
        
        # Store activations
        self.activations = {
            'input': X,
            'hidden_pre': z1,
            'hidden_post': a1,
            'output_pre': z2,
            'output_post': a2
        }
        
        return a2
    
    def backward(self, X, y_true, learning_rate=0.01):
        m = X.shape[0]
        
        # Forward pass
        y_pred = self.forward(X)
        
        # Backward pass
        dz2 = y_pred - y_true
        dW2 = (1/m) * self.activations['hidden_post'].T @ dz2
        db2 = (1/m) * np.sum(dz2, axis=0)
        
        da1 = dz2 @ self.W2.T
        dz1 = da1 * self.activations['hidden_post'] * (1 - self.activations['hidden_post'])
        dW1 = (1/m) * X.T @ dz1
        db1 = (1/m) * np.sum(dz1, axis=0)
        
        # Update weights
        self.W1 -= learning_rate * dW1
        self.b1 -= learning_rate * db1
        self.W2 -= learning_rate * dW2
        self.b2 -= learning_rate * db2
        
        # Return loss
        loss = -np.mean(y_true * np.log(y_pred + 1e-15))
        return loss

def discretize_activations(activations, n_bins=10):
    """Discretize continuous activations for information analysis"""
    # Use quantile-based binning
    discretized = np.zeros_like(activations, dtype=int)
    
    for i in range(activations.shape[1]):
        # Get quantiles for this neuron
        quantiles = np.linspace(0, 1, n_bins + 1)
        thresholds = np.quantile(activations[:, i], quantiles)
        
        # Assign bins
        discretized[:, i] = np.digitize(activations[:, i], thresholds) - 1
        discretized[:, i] = np.clip(discretized[:, i], 0, n_bins - 1)
    
    return discretized

def analyze_network_information(network, X, y, n_bins=10):
    """Analyze information flow in neural network"""
    
    # Get activations
    _ = network.forward(X)
    
    # Discretize for information analysis
    input_discrete = discretize_activations(X, n_bins)
    hidden_discrete = discretize_activations(network.activations['hidden_post'], n_bins)
    
    # Convert labels to discrete
    y_labels = np.argmax(y, axis=1)
    
    # Calculate mutual information
    # Treat each sample as coming from joint distribution
    # This is approximation - true MI requires careful estimation
    
    # I(X; Hidden) - how much input info is preserved in hidden layer
    # Approximate by averaging MI over features/neurons
    mi_input_hidden = 0
    for i in range(hidden_discrete.shape[1]):
        for j in range(input_discrete.shape[1]):
            mi = mutual_info_score(input_discrete[:, j], hidden_discrete[:, i])
            mi_input_hidden += mi
    mi_input_hidden /= (hidden_discrete.shape[1] * input_discrete.shape[1])
    
    # I(Hidden; Y) - how much task info is preserved in hidden layer
    mi_hidden_output = 0
    for i in range(hidden_discrete.shape[1]):
        mi = mutual_info_score(hidden_discrete[:, i], y_labels)
        mi_hidden_output += mi
    mi_hidden_output /= hidden_discrete.shape[1]
    
    # I(X; Y) - mutual information between input and labels
    mi_input_output = 0
    for j in range(input_discrete.shape[1]):
        mi = mutual_info_score(input_discrete[:, j], y_labels)
        mi_input_output += mi
    mi_input_output /= input_discrete.shape[1]
    
    return {
        'mi_input_hidden': mi_input_hidden,
        'mi_hidden_output': mi_hidden_output,
        'mi_input_output': mi_input_output
    }

# Create synthetic dataset
def create_information_dataset():
    """Create dataset for information analysis"""
    np.random.seed(42)
    
    # Create dataset where only some features are relevant
    n_samples = 500
    n_features = 10
    n_relevant = 4  # Only first 4 features are relevant
    
    X = np.random.randn(n_samples, n_features)
    
    # Create labels based on relevant features only
    relevant_features = X[:, :n_relevant]
    # Linear combination with some nonlinearity
    scores = np.sum(relevant_features * [1, -1, 0.5, -0.5], axis=1)
    scores += 0.2 * np.sum(relevant_features**2, axis=1)  # Add nonlinearity
    
    # Create 3 classes based on scores
    y_labels = np.zeros(n_samples, dtype=int)
    y_labels[scores < np.percentile(scores, 33)] = 0
    y_labels[(scores >= np.percentile(scores, 33)) & (scores < np.percentile(scores, 67))] = 1
    y_labels[scores >= np.percentile(scores, 67)] = 2
    
    # One-hot encode
    y = np.eye(3)[y_labels]
    
    return X, y, y_labels

# Demonstrate information bottleneck principle
def demonstrate_information_bottleneck():
    """Demonstrate information bottleneck in neural networks"""
    
    X, y, y_labels = create_information_dataset()
    
    print(f"\nInformation Bottleneck Analysis:")
    print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features, 3 classes")
    print(f"Only first 4 features are relevant for classification")
    
    # Train networks with different hidden layer sizes
    hidden_sizes = [2, 5, 10, 20]
    results = {}
    
    for hidden_size in hidden_sizes:
        print(f"\nTraining network with {hidden_size} hidden units...")
        
        network = InformationAnalysisNetwork(X.shape[1], hidden_size, 3)
        
        # Training loop
        n_epochs = 100
        losses = []
        mi_history = []
        
        for epoch in range(n_epochs):
            loss = network.backward(X, y, learning_rate=0.1)
            losses.append(loss)
            
            # Analyze information every 20 epochs
            if epoch % 20 == 0:
                mi_info = analyze_network_information(network, X, y)
                mi_history.append(mi_info)
        
        # Final evaluation
        predictions = network.forward(X)
        accuracy = np.mean(np.argmax(predictions, axis=1) == y_labels)
        final_mi = analyze_network_information(network, X, y)
        
        results[hidden_size] = {
            'accuracy': accuracy,
            'final_loss': losses[-1],
            'mi_info': final_mi,
            'mi_history': mi_history,
            'losses': losses
        }
        
        print(f"  Final accuracy: {accuracy:.3f}")
        print(f"  Final loss: {losses[-1]:.3f}")
        print(f"  I(X; Hidden): {final_mi['mi_input_hidden']:.3f}")
        print(f"  I(Hidden; Y): {final_mi['mi_hidden_output']:.3f}")
    
    return results

info_results = demonstrate_information_bottleneck()

# Visualize information bottleneck results
def visualize_information_bottleneck():
    """Visualize information bottleneck principle"""
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    hidden_sizes = list(info_results.keys())
    colors = ['blue', 'red', 'green', 'purple']
    
    # Plot 1: Accuracy vs hidden size
    ax1 = axes[0, 0]
    accuracies = [info_results[size]['accuracy'] for size in hidden_sizes]
    
    ax1.plot(hidden_sizes, accuracies, 'bo-', linewidth=2, markersize=8)
    ax1.set_xlabel('Hidden Layer Size')
    ax1.set_ylabel('Accuracy')
    ax1.set_title('Accuracy vs Hidden Layer Size')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Information plane (I(X;Hidden) vs I(Hidden;Y))
    ax2 = axes[0, 1]
    
    for i, size in enumerate(hidden_sizes):
        mi_info = info_results[size]['mi_info']
        ax2.scatter(mi_info['mi_input_hidden'], mi_info['mi_hidden_output'], 
                   s=size*10, color=colors[i], alpha=0.7, label=f'{size} units')
    
    ax2.set_xlabel('I(X; Hidden) - Information about Input')
    ax2.set_ylabel('I(Hidden; Y) - Information about Output')
    ax2.set_title('Information Plane')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Loss curves
    ax3 = axes[1, 0]
    
    for i, size in enumerate(hidden_sizes):
        losses = info_results[size]['losses']
        epochs = range(len(losses))
        ax3.plot(epochs, losses, color=colors[i], linewidth=2, label=f'{size} units')
    
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('Cross-Entropy Loss')
    ax3.set_title('Training Loss Curves')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    ax3.set_yscale('log')
    
    # Plot 4: Information compression analysis
    ax4 = axes[1, 1]
    
    mi_input_hidden = [info_results[size]['mi_info']['mi_input_hidden'] for size in hidden_sizes]
    mi_hidden_output = [info_results[size]['mi_info']['mi_hidden_output'] for size in hidden_sizes]
    
    ax4.bar(np.arange(len(hidden_sizes)) - 0.2, mi_input_hidden, 0.4, 
           label='I(X; Hidden)', alpha=0.7, color='blue')
    ax4.bar(np.arange(len(hidden_sizes)) + 0.2, mi_hidden_output, 0.4, 
           label='I(Hidden; Y)', alpha=0.7, color='red')
    
    ax4.set_xlabel('Hidden Layer Size')
    ax4.set_ylabel('Mutual Information')
    ax4.set_title('Information Compression vs Preservation')
    ax4.set_xticks(range(len(hidden_sizes)))
    ax4.set_xticklabels(hidden_sizes)
    ax4.legend()
    ax4.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

visualize_information_bottleneck()

print(f"\nKey Insights about Information in Neural Networks:")
print(f"• Networks compress input information while preserving task-relevant info")
print(f"• Hidden layer size affects information bottleneck")
print(f"• Too small: May lose important information")
print(f"• Too large: May not compress effectively, prone to overfitting")
print(f"• Information plane shows trade-off between compression and prediction")
print(f"• Cross-entropy loss encourages networks to preserve relevant information")

## 5. Coding Theory and Compression

Information theory provides the foundation for data compression and error correction, with applications in model compression and communication.

In [ ]:
# Coding theory and compression
print("Coding Theory and Compression:")
print("")
print("Shannon's Source Coding Theorem:")
print("• Average code length ≥ H(X) (entropy lower bound)")
print("• Optimal codes achieve average length close to entropy")
print("")
print("Huffman Coding:")
print("• Optimal prefix-free code for given symbol probabilities")
print("• More frequent symbols get shorter codes")
print("")
print("Applications in ML:")
print("• Model compression (pruning, quantization)")
print("• Data preprocessing and storage")
print("• Neural network weight encoding")

class HuffmanNode:
    """Node for Huffman coding tree"""
    def __init__(self, symbol=None, probability=0, left=None, right=None):
        self.symbol = symbol
        self.probability = probability
        self.left = left
        self.right = right
        
    def is_leaf(self):
        return self.left is None and self.right is None

def build_huffman_tree(symbols, probabilities):
    """Build Huffman coding tree"""
    import heapq
    
    # Create leaf nodes
    heap = []
    for i, (symbol, prob) in enumerate(zip(symbols, probabilities)):
        node = HuffmanNode(symbol=symbol, probability=prob)
        heapq.heappush(heap, (prob, i, node))  # Use index to break ties
    
    # Build tree
    node_counter = len(symbols)
    while len(heap) > 1:
        # Get two nodes with smallest probabilities
        prob1, _, node1 = heapq.heappop(heap)
        prob2, _, node2 = heapq.heappop(heap)
        
        # Create internal node
        merged_prob = prob1 + prob2
        merged_node = HuffmanNode(probability=merged_prob, left=node1, right=node2)
        
        heapq.heappush(heap, (merged_prob, node_counter, merged_node))
        node_counter += 1
    
    return heap[0][2] if heap else None

def generate_huffman_codes(root):
    """Generate Huffman codes from tree"""
    if root is None:
        return {}
    
    codes = {}
    
    def traverse(node, code=""):
        if node.is_leaf():
            codes[node.symbol] = code if code else "0"  # Handle single symbol case
        else:
            if node.left:
                traverse(node.left, code + "0")
            if node.right:
                traverse(node.right, code + "1")
    
    traverse(root)
    return codes

def calculate_average_length(symbols, probabilities, codes):
    """Calculate average code length"""
    return sum(prob * len(codes[symbol]) for symbol, prob in zip(symbols, probabilities))

# Demonstrate Huffman coding
def demonstrate_huffman_coding():
    """Demonstrate Huffman coding with different distributions"""
    
    # Example 1: English letter frequencies (simplified)
    english_letters = ['E', 'T', 'A', 'O', 'I', 'N', 'S', 'H', 'R']
    english_probs = [0.127, 0.091, 0.082, 0.075, 0.070, 0.067, 0.063, 0.061, 0.060]
    
    # Example 2: Skewed distribution
    skewed_symbols = ['A', 'B', 'C', 'D']
    skewed_probs = [0.7, 0.2, 0.08, 0.02]
    
    # Example 3: Uniform distribution
    uniform_symbols = ['A', 'B', 'C', 'D']
    uniform_probs = [0.25, 0.25, 0.25, 0.25]
    
    examples = [
        (english_letters, english_probs, "English Letters"),
        (skewed_symbols, skewed_probs, "Skewed Distribution"),
        (uniform_symbols, uniform_probs, "Uniform Distribution")
    ]
    
    print(f"\nHuffman Coding Analysis:")
    print(f"{'Distribution':<20} {'Entropy':<10} {'Avg Length':<12} {'Efficiency':<12}")
    print("-" * 60)
    
    huffman_results = {}
    
    for symbols, probs, name in examples:
        # Build Huffman tree and generate codes
        tree = build_huffman_tree(symbols, probs)
        codes = generate_huffman_codes(tree)
        
        # Calculate metrics
        H = entropy(probs)
        avg_length = calculate_average_length(symbols, probs, codes)
        efficiency = H / avg_length if avg_length > 0 else 0
        
        huffman_results[name] = {
            'symbols': symbols,
            'probabilities': probs,
            'codes': codes,
            'entropy': H,
            'avg_length': avg_length,
            'efficiency': efficiency
        }
        
        print(f"{name:<20} {H:<10.3f} {avg_length:<12.3f} {efficiency:<12.3f}")
    
    # Show actual codes for skewed example
    print(f"\nHuffman Codes for Skewed Distribution:")
    skewed_codes = huffman_results["Skewed Distribution"]['codes']
    for symbol, prob in zip(skewed_symbols, skewed_probs):
        code = skewed_codes[symbol]
        print(f"  {symbol}: {code} (length {len(code)}, probability {prob})")
    
    return huffman_results

huffman_results = demonstrate_huffman_coding()

# Model compression example
def demonstrate_model_compression():
    """Demonstrate model compression using information theory principles"""
    
    # Create a simple "model" with weights
    np.random.seed(42)
    original_weights = np.random.randn(1000) * 2  # 1000 weights
    
    print(f"\nModel Compression Example:")
    print(f"Original model: 1000 weights (float32)")
    print(f"Original size: {1000 * 4} bytes (32 bits per weight)")
    
    # Compression technique 1: Quantization
    def quantize_weights(weights, n_bits=8):
        """Quantize weights to n_bits precision"""
        min_val, max_val = weights.min(), weights.max()
        n_levels = 2**n_bits
        
        # Quantize
        scale = (max_val - min_val) / (n_levels - 1)
        quantized = np.round((weights - min_val) / scale)
        quantized = np.clip(quantized, 0, n_levels - 1)
        
        # Dequantize for error calculation
        dequantized = quantized * scale + min_val
        
        return quantized.astype(int), dequantized, (min_val, max_val, scale)
    
    # Compression technique 2: Sparsification (set small weights to zero)
    def sparsify_weights(weights, threshold_percentile=50):
        """Set weights below threshold to zero"""
        threshold = np.percentile(np.abs(weights), threshold_percentile)
        sparse_weights = weights.copy()
        sparse_weights[np.abs(sparse_weights) < threshold] = 0
        return sparse_weights
    
    # Apply compression techniques
    compression_results = {}
    
    # Original
    compression_results['Original'] = {
        'weights': original_weights,
        'size_bits': len(original_weights) * 32,
        'mse': 0,
        'sparsity': 0
    }
    
    # Quantization to different bit widths
    for n_bits in [8, 4, 2]:
        quantized, dequantized, params = quantize_weights(original_weights, n_bits)
        mse = np.mean((original_weights - dequantized)**2)
        
        # Calculate actual storage size (quantized values + parameters)
        size_bits = len(quantized) * n_bits + 3 * 32  # quantized + (min, max, scale)
        
        compression_results[f'{n_bits}-bit Quantized'] = {
            'weights': dequantized,
            'size_bits': size_bits,
            'mse': mse,
            'sparsity': 0
        }
    
    # Sparsification
    for percentile in [50, 70, 90]:
        sparse_weights = sparsify_weights(original_weights, percentile)
        mse = np.mean((original_weights - sparse_weights)**2)
        sparsity = np.mean(sparse_weights == 0)
        
        # Estimate compressed size (only store non-zero weights + indices)
        n_nonzero = np.sum(sparse_weights != 0)
        size_bits = n_nonzero * 32 + n_nonzero * 10  # weight + index (assuming 10 bits for index)
        
        compression_results[f'{percentile}th% Sparse'] = {
            'weights': sparse_weights,
            'size_bits': size_bits,
            'mse': mse,
            'sparsity': sparsity
        }
    
    # Print results
    print(f"\nCompression Results:")
    print(f"{'Method':<15} {'Size (KB)':<12} {'Compression':<12} {'MSE':<12} {'Sparsity':<10}")
    print("-" * 70)
    
    original_size = compression_results['Original']['size_bits']
    
    for method, result in compression_results.items():
        size_kb = result['size_bits'] / (8 * 1024)
        compression_ratio = original_size / result['size_bits']
        mse = result['mse']
        sparsity = result['sparsity']
        
        print(f"{method:<15} {size_kb:<12.3f} {compression_ratio:<12.2f} {mse:<12.6f} {sparsity:<10.2f}")
    
    return compression_results

compression_results = demonstrate_model_compression()

# Visualize coding theory results
def visualize_coding_theory():
    """Visualize coding theory and compression results"""
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Plot 1: Huffman coding efficiency
    ax1 = axes[0, 0]
    
    distributions = list(huffman_results.keys())
    entropies = [huffman_results[dist]['entropy'] for dist in distributions]
    avg_lengths = [huffman_results[dist]['avg_length'] for dist in distributions]
    
    x_pos = np.arange(len(distributions))
    width = 0.35
    
    ax1.bar(x_pos - width/2, entropies, width, label='Entropy (Lower Bound)', alpha=0.7)
    ax1.bar(x_pos + width/2, avg_lengths, width, label='Huffman Avg Length', alpha=0.7)
    
    ax1.set_xlabel('Distribution Type')
    ax1.set_ylabel('Bits per Symbol')
    ax1.set_title('Huffman Coding Efficiency')
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels([dist.split()[0] for dist in distributions])
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Plot 2: Code length vs probability
    ax2 = axes[0, 1]
    
    # Use skewed distribution example
    skewed_data = huffman_results['Skewed Distribution']
    symbols = skewed_data['symbols']
    probs = skewed_data['probabilities']
    codes = skewed_data['codes']
    
    code_lengths = [len(codes[symbol]) for symbol in symbols]
    optimal_lengths = [-np.log2(p) for p in probs]  # Optimal code length
    
    ax2.scatter(probs, code_lengths, s=100, color='red', alpha=0.7, label='Huffman Code Length')
    ax2.plot(probs, optimal_lengths, 'b--', linewidth=2, label='Optimal Length (-log₂p)')
    
    for i, symbol in enumerate(symbols):
        ax2.annotate(symbol, (probs[i], code_lengths[i]), 
                    xytext=(5, 5), textcoords='offset points')
    
    ax2.set_xlabel('Symbol Probability')
    ax2.set_ylabel('Code Length (bits)')
    ax2.set_title('Code Length vs Probability')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Model compression trade-offs
    ax3 = axes[1, 0]
    
    methods = list(compression_results.keys())
    compression_ratios = []
    mses = []
    
    original_size = compression_results['Original']['size_bits']
    
    for method in methods:
        if method != 'Original':
            ratio = original_size / compression_results[method]['size_bits']
            mse = compression_results[method]['mse']
            compression_ratios.append(ratio)
            mses.append(mse)
    
    scatter = ax3.scatter(compression_ratios, mses, s=100, alpha=0.7, c=range(len(compression_ratios)), cmap='viridis')
    
    # Add labels
    for i, method in enumerate(methods[1:]):  # Skip 'Original'
        ax3.annotate(method.replace(' ', '\n'), (compression_ratios[i], mses[i]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    ax3.set_xlabel('Compression Ratio')
    ax3.set_ylabel('Mean Squared Error')
    ax3.set_title('Compression vs Accuracy Trade-off')
    ax3.set_yscale('log')
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Weight distribution before and after compression
    ax4 = axes[1, 1]
    
    original = compression_results['Original']['weights']
    quantized_4bit = compression_results['4-bit Quantized']['weights']
    sparse_70 = compression_results['70th% Sparse']['weights']
    
    ax4.hist(original, bins=50, alpha=0.5, label='Original', density=True)
    ax4.hist(quantized_4bit, bins=50, alpha=0.5, label='4-bit Quantized', density=True)
    ax4.hist(sparse_70[sparse_70 != 0], bins=30, alpha=0.5, label='70% Sparse (non-zero)', density=True)
    
    ax4.set_xlabel('Weight Value')
    ax4.set_ylabel('Density')
    ax4.set_title('Weight Distributions After Compression')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

visualize_coding_theory()

print(f"\nKey Insights about Coding Theory and Compression:")
print(f"• Huffman coding achieves near-optimal compression (close to entropy)")
print(f"• More skewed distributions compress better")
print(f"• Model compression involves trade-offs between size and accuracy")
print(f"• Quantization reduces precision but maintains structure")
print(f"• Sparsification can achieve high compression with careful thresholding")
print(f"• Information theory provides bounds on achievable compression rates")

## Summary: Theoretical Foundations and Modern Applications

This comprehensive treatment of information theory provides the rigorous mathematical foundations essential for understanding modern machine learning algorithms. The integration of measure theory, differential geometry, and algorithmic information theory enables deep insights into learning, compression, and optimization.

### Core Theoretical Achievements:

**1. Measure-Theoretic Information Theory:**
- **Shannon Entropy**: Rigorous foundation using probability measures and integration theory
- **f-Divergences**: Unified framework for measuring distributional differences
- **Information Inequalities**: Data processing, submodularity, and log-Sobolev inequalities
- **Maximum Entropy Principle**: Variational characterization of optimal distributions

**2. Advanced Information Measures:**
- **Rényi Entropies**: Generalized entropy spectrum and order statistics
- **Rate-Distortion Theory**: Fundamental limits of lossy compression
- **Information Bottleneck**: Optimal trade-off between compression and prediction
- **Channel Capacity**: Fundamental limits of communication and information transmission

**3. Algorithmic Information Theory:**
- **Kolmogorov Complexity**: Absolute measure of string randomness and compressibility
- **Minimum Description Length**: Model selection via compression-based inference
- **Universal Coding**: Optimal codes for unknown sources
- **PAC-Bayesian Theory**: Information-theoretic generalization bounds

**4. Information Geometry:**
- **Statistical Manifolds**: Riemannian geometry of probability distributions
- **Fisher Information Metric**: Natural metric for statistical inference
- **Dual Connections**: Amari's α-connections and dually flat structures
- **Natural Gradients**: Geometrically motivated optimization algorithms

### Deep Connections to Machine Learning:

**Statistical Learning Theory:**
- **Generalization Bounds**: Information-theoretic complexity measures in PAC-Bayes theory
- **Model Selection**: MDL principle for balancing fit and complexity
- **Feature Selection**: Mutual information maximization and conditional independence testing
- **Representation Learning**: Information bottleneck principle in deep learning

**Deep Learning Foundations:**
- **Loss Functions**: Cross-entropy and f-divergence losses
- **Natural Gradients**: Information-geometric optimization for neural networks
- **Variational Inference**: ELBO optimization using KL divergence
- **Information Bottleneck**: Deep learning as hierarchical information processing

**Modern Applications:**
- **Generative Models**: VAEs with information-theoretic objectives
- **Optimal Transport**: Wasserstein information geometry and GANs
- **Compression**: Neural network pruning and quantization using rate-distortion theory
- **Causal Inference**: Information-theoretic measures of causal strength

### Research Frontiers:

**Theoretical Machine Learning:**
- **Deep Information Bottleneck**: Understanding compression vs. generalization trade-offs
- **Non-parametric Information Theory**: Information measures for infinite-dimensional spaces
- **Quantum Information**: Extending classical information theory to quantum systems
- **Causal Information Theory**: Information measures respecting causal structure

**Algorithmic Advances:**
- **Efficient Estimation**: Scalable algorithms for mutual information estimation
- **Natural Gradients**: Second-order optimization using information geometry
- **Differentiable Information**: Gradient-based optimization of information-theoretic objectives
- **Multi-modal Learning**: Information fusion across different modalities

**Emerging Paradigms:**
- **Federated Information**: Information theory for distributed learning
- **Privacy-Preserving ML**: Differential privacy and information-theoretic privacy measures
- **Continual Learning**: Information-theoretic approaches to catastrophic forgetting
- **Meta-Learning**: Information-efficient few-shot learning algorithms

### Critical Research Questions:

1. **Information vs. Generalization**: How does information compression in neural networks relate to generalization performance?

2. **Efficient Estimation**: Can we develop scalable, consistent estimators for high-dimensional mutual information?

3. **Geometric Learning**: How can information geometry improve optimization and generalization in deep learning?

4. **Causal Information**: What are the fundamental information-theoretic limits of causal discovery and inference?

This theoretical foundation enables rigorous analysis of learning algorithms, providing both mathematical tools for algorithm design and fundamental limits for what is achievable. Information theory unifies diverse ML concepts under a coherent mathematical framework.

**Next Steps**: With this information theory foundation, you are prepared for advanced topics including optimal transport theory, quantum machine learning, causal inference, information-geometric deep learning, and modern generative modeling techniques.

**Research Impact**: These information-theoretic principles directly inform cutting-edge research in deep learning theory, explaining phenomena like the lottery ticket hypothesis, implicit regularization in overparameterized models, and the effectiveness of various architectural choices through the lens of information processing and compression.